In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module=".*")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json

### ID3 Algorithm

In [ ]:
class DecisionTree:
    """
    Decision Tree Classifier Implementation.
    This class implements a simple decision tree classifier using the ID3 algorithm.
    The algorithm follows a top-down, greedy search approach
    through the given dataset to construct a decision tree.It begins
    with the entire dataset and divides it into subsets based on the
    attribute that maximizes the Information Gain (or Information Gain Ratio),
    intending to efficiently classify the instances at each node of
    the tree.
    """

    def __init__(self, max_depth=None) -> None:
        self.max_depth = max_depth
        self.tree = None
        
    def _entropy(self, y) -> float:
        """
            Calculate the entropy of the class labels.
        """
        class_labels, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return -np.sum(probabilities * np.log2(probabilities + 1e-10))

    def fit(self, X, y) -> None:
        """
            Fit the decision tree classifier to the training data.
        """
        self.tree = self._build_tree(X, y)

    def predict(self, X) -> list:
        """
            Predict class labels for the input data.
        """
        return [self._predict_instance(x, self.tree) for x in X]
    
    def _best_feature_to_split(self, X, y) -> int:
        """
            Find the feature that provides the best split for the dataset.
        """
        n_samples, n_features = X.shape
        best_gain = -1
        best_feature = -1

        # Calculate the entropy of the current dataset
        base_entropy = self._entropy(y)
        for feature in range(n_features):
            # Get unique values for the feature
            unique_values = np.unique(X[:, feature])
            new_entropy = 0.0
            # Calculate the new entropy after splitting on this feature
            for value in unique_values:
                subset_y = y[X[:, feature] == value]
                prob = len(subset_y) / n_samples
                new_entropy += prob * self._entropy(subset_y)
            # Calculate information gain
            gain = base_entropy - new_entropy
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
        return best_feature
    
    def _build_tree(self, X, y, depth=0):
        """
            Recursively build the decision tree.
        """
        n_samples, n_features = X.shape
        unique_classes, counts = np.unique(y, return_counts=True)
        most_common_class = unique_classes[np.argmax(counts)]
        # Stop if all samples are of the same class or max depth is reached
        if len(unique_classes) == 1 or (self.max_depth is not None and depth >= self.max_depth):
            return most_common_class
        # Find the best feature to split on
        best_feature = self._best_feature_to_split(X, y)
        tree = {best_feature: {}}
        # Split the dataset on the best feature
        for value in np.unique(X[:, best_feature]):
            indices = np.where(X[:, best_feature] == value)[0]
            subtree = self._build_tree(X[indices], y[indices], depth + 1)
            tree[best_feature][value] = subtree
        return tree
    
    def _predict_instance(self, x, tree) -> int:
        """ 
            if not isinstance(tree, dict):a single instance using the decision tree.
        """
        if not isinstance(tree, dict):
            return tree

        feature = next(iter(tree))
        feature_value = x[feature]
        if feature_value in tree[feature]:
            subtree = tree[feature][feature_value]
            return self._predict_instance(x, subtree)
        else:
            return None # Default to None if the feature value is not found in the tree
    
    def predict_proba(self, X) -> list:
        """
            Predict class probabilities for the input data.
        """
        predictions = self.predict(X)
        class_labels, counts = np.unique(predictions, return_counts=True)
        probabilities = counts / len(predictions)
        return {label: prob for label, prob in zip(class_labels, probabilities)}
    
    def score(self, X, y) -> dict:
        """
            Calculate the accuracy of the model on the test data.
        """
        predictions = self.predict(X)
        accuracy = np.mean(predictions == y)
        return {
            'accuracy': accuracy,
            'mse': mean_squared_error(y, predictions),
            'mae': mean_absolute_error(y, predictions),
            'r2': r2_score(y, predictions)
        }
        
    def plot_tree(self, tree=None, feature_names=None, class_names=None, filename='tree.png'):
        """"
            Plot the decision tree using matplotlib.
        """
        if tree is None:
            tree = self.tree
        plt.figure(figsize=(12, 8))
        plt.title("Decision Tree")
        plt.axis('off')
        def plot_node(node, depth=0):
            if isinstance(node, dict):
                for feature, branches in node.items():
                    for value, subtree in branches.items():
                        plt.text(depth, -depth, f"{feature} = {value}", fontsize=12)
                        plot_node(subtree, depth + 1)
            else:
                plt.text(depth, -depth, f"Class: {node}", fontsize=12)
        plot_node(tree)
        plt.savefig(filename)
        plt.close()
    
    def save(self, filename):
        """
            Save the decision tree model to a file.
        """
        with open(filename, 'wb') as f:
            pickle.dump(self, f)